# Telecom Customer Churn — Data Quality, Feature Engineering & Modelling

**Competition submission notebook** · Mustafa Al-Rouby

---

## Business context

Customer churn is the central economics problem in telecom. Acquiring a replacement
customer costs up to **5x more** than retaining an existing one, so even a small
improvement in retention targeting is worth more than a large improvement in
acquisition.

## Objective

Predict whether a customer will churn in the next billing cycle (binary
classification), and align the evaluation with business impact rather than pure
statistical accuracy.

## What this notebook covers

| Section | Question it answers |
|---|---|
| 1. Data loading | What are we working with? |
| 2. Data profiling | Is the data trustworthy? |
| 3. Cleaning decisions | What did we change, and why? |
| 4. Auditing the brief's claims | Do the documented defects actually exist? |
| 5. Feature engineering | What signals did we construct? |
| 6. Model selection | Which models, and why? |
| 7. Validation approach | How do we know the score is real? |
| 8. Business impact | What is this worth in money? |
| 9. Bonus challenges | SHAP, drift, cost-sensitive learning, survival |
| 10. Conclusion | What should the business actually do? |

> **Short on time?** Read section 8. The honest headline is that this dataset does
> not support churn targeting, and the notebook quantifies exactly how much value
> the model fails to capture.

---
# 1. Data loading

Three tables: one row per customer in `customer_info` and `churn_labels`, one row
per customer per month in `usage_data`.

In [1]:
import json
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

ci = pd.read_csv("../data/customer_info.csv")
ud = pd.read_csv("../data/usage_data.csv")
cl = pd.read_csv("../data/churn_labels.csv")

for name, d in [("customer_info", ci), ("usage_data", ud), ("churn_labels", cl)]:
    print(f"{name:16s} {d.shape[0]:>7,} rows x {d.shape[1]} cols")

print(f"\nunique customers: {ci.CustomerID.nunique():,}")
print(f"churn rate      : {cl.Churn.mean():.4%}  ({int(cl.Churn.sum())} of {len(cl):,})")
print(f"imbalance ratio : 1 : {(1 - cl.Churn.mean()) / cl.Churn.mean():.1f}")

customer_info     10,200 rows x 7 cols
usage_data       120,000 rows x 6 cols
churn_labels      10,000 rows x 2 cols

unique customers: 10,000
churn rate      : 7.8600%  (786 of 10,000)
imbalance ratio : 1 : 11.7


---
# 2. Data profiling — the 11 defects

A churn model built on unvalidated data produces confident answers nobody can
trust, so the audit came first. Every finding is computed in code and stored in
`outputs/data_quality_register.csv`.

In [2]:
reg = pd.read_csv("../outputs/data_quality_register.csv")
print(f"{len(reg)} issues found\n")
print(reg[["issue_id", "fields", "issue", "affected_rows", "affected_pct",
           "severity", "resolution"]].to_string(index=False))

11 issues found

issue_id                   fields                                                                                                                          issue  affected_rows  affected_pct            severity           resolution
   DQ-01                      all                                                                                               Exact duplicate customer records            195         1.950                High           auto-fixed
   DQ-02               SignupDate                                                                  Conflicting duplicate records (same ID, different SignupDate)             10         0.100                High   flagged-for-review
   DQ-03                      Age                                                                                  Missing values (missing completely at random)           3500        35.000                High auto-fixed-with-flag
   DQ-04           MonthlyCharges                          

**Reading the register.** Two findings change how the data can be used at all:

- **DQ-11** — the target label is uninformative (proved in section 7)
- **DQ-07** — 2,278 usage rows are dated *before* the customer signed up, so any
  tenure-derived feature is unreliable for 751 customers

The rest are repairable. Note the `resolution` column: only logically impossible
values were auto-repaired; judgement calls were flagged for a human.

---
# 3. Data cleaning decisions

## Governing principle: nothing is silently fixed

Invalid values become `NULL` in a companion `*_clean` column **and** raise a boolean
`dq_*` flag. A downstream analyst can always see which rows were touched. This
matters because a silently-cleaned dataset hides the reason a metric moved.

In [3]:
clean = pd.read_csv("../outputs/clean/customer_info_clean.csv")
print(f"raw rows     : {len(ci):,}")
print(f"cleaned rows : {len(clean):,}   ({len(ci) - len(clean):,} duplicate rows removed)")
print(f"unique ID    : {clean.CustomerID.is_unique}\n")

for col, desc in {
    "dq_dup_conflict": "conflicting duplicate SignupDate (DQ-02)",
    "dq_charge_negative": "negative monthly charge (DQ-05)",
    "dq_charge_contaminated": "value from contaminant distribution (DQ-04)",
    "dq_charge_implausible_low": "implausibly low charge (DQ-06)",
}.items():
    print(f"{desc:44s} {int(clean[col].sum()):5d} rows")
print(f"{'MonthlyCharges set to NULL':44s} "
      f"{int(clean.MonthlyCharges_clean.isna().sum()):5d} rows")

raw rows     : 10,200
cleaned rows : 10,000   (200 duplicate rows removed)
unique ID    : True

conflicting duplicate SignupDate (DQ-02)         5 rows
negative monthly charge (DQ-05)                  5 rows
value from contaminant distribution (DQ-04)     49 rows
implausibly low charge (DQ-06)                  24 rows
MonthlyCharges set to NULL                      54 rows


### The decisions, and the reasoning behind each

| Defect | Decision | Why |
|---|---|---|
| 195 exact duplicate rows | **Drop** | Byte-identical rows carry no information |
| 5 conflicting duplicates | **Keep first + flag + quarantine** | All 5 disagree only on `SignupDate`, and every pair is an exact month/day swap — a `DD/MM` vs `MM/DD` parsing ambiguity. Neither value is trustworthy and the fault is systemic, so it goes to the data owner rather than being patched |
| `Age` 35.00% missing | **Impute median + flag, never drop** | The missing rate is flat across Gender (0.341–0.359), Region (0.347–0.353), ContractType (0.346–0.354) and Churn (0.350 / 0.356) — all inside ±3 standard errors. That is MCAR: dropping rows would discard 35% of the base and fix nothing |
| 49 contaminant charges | **Set `NULL` + flag** | A 2-component Gaussian mixture isolates them (weight 0.0050, mean 257.42, ΔBIC = 6,152). A log-normal fitted to the clean body predicts 0.79 rows above 200; 38 are observed |
| 5 negative charges | **Set `NULL` + flag** | Logically impossible |
| 24 implausibly low charges | **Flag only** | Suspicious but not impossible — a human decides |
| 2,278 pre-signup usage rows | **Flag, exclude from tenure metrics** | No algorithm can recover the truth; needs the data owner |
| 58 data-usage zeros | **Flag as censored** | Those rows average 518.6 call-minutes vs 492.4 overall, so the customers were active. The 0 is a recording floor |

### Why missing Age was not imputed with a model

With MCAR missingness there is no information to impute *from* — a model would
reproduce the marginal distribution while adding false confidence. Median
imputation behind a flag is the honest choice, and the flag lets the learner test
whether missingness itself is informative (it is not: churn is 35.0% vs 35.6% by
missingness status).

---
# 4. Auditing the brief's own claims

The brief states two things that deserve verification rather than acceptance.

In [4]:
# Claim 1: "SignupDate (inconsistent formats included)"
raw = ci.SignupDate.astype(str).str.replace(r"\d", "9", regex=True)
print("SignupDate string shapes:")
print(raw.value_counts().to_string())
print(f"\ncontains a '/': {ci.SignupDate.astype(str).str.contains('/').sum()}")
print(f"unparseable   : {int(pd.to_datetime(ci.SignupDate, errors='coerce').isna().sum())}")
print()
print("-> every value is uniform ISO 9999-99-99. There are NO format inconsistencies.")
print("   The real date defect is the 5 duplicate-conflict rows with swapped month/day.")

SignupDate string shapes:
SignupDate
9999-99-99    10200

contains a '/': 0
unparseable   : 0

-> every value is uniform ISO 9999-99-99. There are NO format inconsistencies.
   The real date defect is the 5 duplicate-conflict rows with swapped month/day.


In [5]:
# Claim 2: "Churn patterns are behaviourally embedded (e.g. usage decline before churn)"
uni = pd.read_csv("../outputs/trend_univariate.csv")
print("Trend features tested for a churn signal (decline = negative slope, ratio < 1):\n")
print(uni.to_string(index=False))
print(f"\nfeatures with p < 0.05: {(uni.p < 0.05).sum()} of {len(uni)}")
print()
print("-> no trend feature separates churners. The claimed usage-decline signal is")
print("   not present in the delivered data.")

Trend features tested for a churn signal (decline = negative slope, ratio < 1):

              feature         t        p         r
comp_last_minus_first  1.448270 0.147878  0.014125
       comp_halfratio -0.635255 0.525460 -0.006963
       call_halfratio  0.560963 0.574959  0.005621
           data_slope -0.490499 0.623899 -0.005075
call_last_minus_first -0.470792 0.637901 -0.004783
        sms_halfratio -0.445999 0.655708 -0.005225
 sms_last_minus_first -0.442986 0.657879 -0.004331
            sms_slope -0.402169 0.687653 -0.004033
           call_slope  0.312705 0.754576  0.003176
           comp_slope -0.136254 0.891650 -0.001343
data_last_minus_first -0.120073 0.904452 -0.001206
       data_halfratio  0.056411 0.955026  0.000563

features with p < 0.05: 0 of 12

-> no trend feature separates churners. The claimed usage-decline signal is
   not present in the delivered data.


**This matters.** A submission that assumed the brief was accurate would have spent
its effort engineering decline features that cannot work, and could have reported a
model as successful on the strength of a lucky validation split. Verifying the brief
against the bytes is what makes the rest of this notebook trustworthy.

---
# 5. Feature engineering strategy

66 features per customer, in six documented families. The trend family is
first-class rather than an afterthought, because the brief's stated mechanism is a
*decline* — and a standard deviation cannot express direction (a steady rise and a
steady decline have identical SDs).

In [6]:
feats = pd.read_csv("../outputs/features.csv")
print(f"feature table: {feats.shape[0]:,} customers x {feats.shape[1]} columns\n")

families = {
    "Demographic": ["Age_imputed", "age_missing", "is_male"],
    "Contract": ["MonthlyCharges_clean", "charge_invalid", "tenure_months",
                 "signup_year", "signed_up_in_window"],
    "Usage level": ["call_mean", "data_median", "sms_max", "comp_min"],
    "Usage volatility": ["call_sd", "data_cv", "sms_sd", "comp_cv"],
    "Usage trend": ["call_slope", "data_halfratio", "sms_rel_slope",
                    "comp_last_minus_first"],
    "Engagement": ["n_complaint_months", "data_per_call_min",
                   "complaints_per_call_hour"],
}
for fam, cols in families.items():
    present = [c for c in cols if c in feats.columns]
    print(f"{fam:18s} e.g. {', '.join(present)}")
print(f"\ntotal numeric features: {feats.select_dtypes('number').shape[1]}")

feature table: 10,000 customers x 69 columns

Demographic        e.g. Age_imputed, age_missing, is_male
Contract           e.g. MonthlyCharges_clean, charge_invalid, tenure_months, signup_year, signed_up_in_window
Usage level        e.g. call_mean, data_median, sms_max, comp_min
Usage volatility   e.g. call_sd, data_cv, sms_sd, comp_cv
Usage trend        e.g. call_slope, data_halfratio, sms_rel_slope, comp_last_minus_first
Engagement         e.g. n_complaint_months, data_per_call_min, complaints_per_call_hour

total numeric features: 66


### Rationale for the trend family

For each of the four usage metrics (`CallMinutes`, `DataUsageGB`, `SMSCount`,
`Complaints`) we compute:

| Feature | Definition | Why |
|---|---|---|
| `*_slope` | OLS slope across the 12 months | The literal "decline" mechanism |
| `*_rel_slope` | slope ÷ mean | Scale-free, comparable across customers |
| `*_halfratio` | mean(months 10–12) ÷ mean(months 1–3) | Robust end-vs-start comparison |
| `*_last_minus_first` | month 12 − month 1 | Simple directional change |
| `*_trend_r2` | R² of the linear fit | Is the trend consistent or noise? |
| `*_declining` | slope < 0 | Binary flag for the mechanism |

Plus engagement ratios (`data_per_call_min`, `complaints_per_call_hour`) that
normalise behaviour by activity level, so a low-usage customer is not mistaken for
a disengaging one.

### Leakage control

Every feature is computed per customer from that customer's own 12 months. No
target statistics, no cross-customer aggregates, no target-derived encodings. The
`dq_*` flags are deliberately included: they let the model test whether *being a
defective record* correlates with churn (it does not, but testing it is the point).

---
# 6. Model selection reasoning

Five candidates, spanning the bias–variance spectrum and including an explicit
no-skill baseline:

| Model | Why it is in the comparison |
|---|---|
| **Dummy (base rate)** | The honest floor. Anything that cannot beat it is not a model |
| **Logistic regression** | Linear, interpretable, well-calibrated by construction; the right default for a mostly-categorical tabular problem |
| **Logistic (`class_weight='balanced'`)** | A first, cheap cost-sensitive response to the 1:12 imbalance |
| **Random forest** | Captures non-linearity and interactions without scaling, and is robust to irrelevant features — which matters when most of the 66 features are noise |
| **Gradient boosting** | Usually the strongest tabular learner, and it can find interactions a linear model cannot |

In [7]:
s = json.load(open("../outputs/model_summary.json"))
print(f"design matrix : {s['n_rows']:,} rows x {s['n_features']} features")
print(f"base rate     : {s['base_rate']:.4%}\n")
print(f"{'model':26s} {'ROC-AUC':>9s} {'PR-AUC':>9s} {'Brier':>8s}")
print(f"{'(no-skill reference)':26s} {'0.5000':>9s} {s['base_rate']:9.4f} "
      f"{s['base_rate'] * (1 - s['base_rate']):8.4f}")
for name, m in s["models"].items():
    print(f"{name:26s} {m['roc_auc']:9.4f} {m['pr_auc']:9.4f} {m['brier']:8.4f}")

design matrix : 10,000 rows x 70 features
base rate     : 7.8600%

model                        ROC-AUC    PR-AUC    Brier
(no-skill reference)          0.5000    0.0786   0.0724
Dummy (base rate)             0.4994    0.0785   0.0724
Logistic regression           0.4964    0.0808   0.0730
Logistic (balanced)           0.4949    0.0808   0.2476
Random forest                 0.5010    0.0815   0.0734
Gradient boosting             0.5181    0.0808   0.0748


### Why PR-AUC is reported next to ROC-AUC

With 7.86% positives, ROC-AUC flatters a model — a useless classifier can still
score near 0.5. **Average precision** uses the base rate as its no-skill reference,
so `PR-AUC ≈ base rate` is the signature of no signal. The table above should be
read that way.

---
# 7. Validation approach

## There is no held-out test set

`churn_labels.csv` covers all 10,000 customers, so there is nothing to hold out and
in-sample metrics would be meaningless. Instead:

1. **Stratified 5-fold cross-validation** preserves the 7.86% positive rate in
   every fold.
2. **Out-of-fold predictions** — each customer is scored by a model trained without
   them. Every metric here, and the submitted probabilities themselves, are
   out-of-fold.
3. **A shuffled-label null** establishes what "no signal" looks like at this sample
   size, so a score is judged against noise rather than against 0.5.

In [8]:
print("RANDOMISATION TEST — is the score real?")
print(f"best model          : {s['best_model']}  ROC-AUC = {s['best_roc_auc']:.4f}")
print(f"shuffled-label null : {s['null_band'][0]:.4f} - {s['null_band'][1]:.4f}")
print(f"share of null draws >= best model: {s['null_share_ge_best']:.1%}\n")
if s["null_share_ge_best"] < 0.05:
    print("VERDICT: real signal present.")
else:
    print("VERDICT: INDISTINGUISHABLE from random labels.")
    print("         The model's ranking carries no information about churn.")

RANDOMISATION TEST — is the score real?
best model          : Gradient boosting  ROC-AUC = 0.5181
shuffled-label null : 0.4772 - 0.5198
share of null draws >= best model: 10.0%

VERDICT: INDISTINGUISHABLE from random labels.
         The model's ranking carries no information about churn.


In [9]:
print("DECILE LIFT — the operational question: who do we contact first?\n")
lift = pd.read_csv("../outputs/decile_lift.csv")
lift["churn_rate"] = (lift.churn_rate * 100).round(2)
lift["cum_recall"] = (lift.cum_recall * 100).round(1)
print(lift.to_string(index=False))
print()
print(f"Top-decile lift: {s['top_decile_lift']:.2f}x")
print("-> a lift at or below 1.0 means the 'highest risk' decile churns no more often")
print("   than a random sample. There is no one to prioritise.")

DECILE LIFT — the operational question: who do we contact first?

 decile    n  churners  churn_rate     lift  cum_recall
      1 1000        75         7.5 0.954198         9.5
      2 1000        86         8.6 1.094148        20.5
      3 1000        87         8.7 1.106870        31.6
      4 1000        85         8.5 1.081425        42.4
      5 1000        85         8.5 1.081425        53.2
      6 1000        75         7.5 0.954198        62.7
      7 1000        75         7.5 0.954198        72.3
      8 1000        78         7.8 0.992366        82.2
      9 1000        76         7.6 0.966921        91.9
     10 1000        64         6.4 0.814249       100.0

Top-decile lift: 0.95x
-> a lift at or below 1.0 means the 'highest risk' decile churns no more often
   than a random sample. There is no one to prioritise.


In [10]:
print("CALIBRATION — are the submitted probabilities honest?\n")
cal = pd.read_csv("../outputs/calibration.csv")
print(cal.round(4).to_string(index=False))
print()
print(f"predicted mean {s['submitted_mean_prob']:.4f} vs base rate {s['base_rate']:.4f}")
print(f"Brier {s['submitted_brier']:.4f} vs no-skill "
      f"{s['base_rate'] * (1 - s['base_rate']):.4f}")
print("-> the probabilities sit at the base rate, which is the correct answer when")
print("   there is nothing to predict.")

CALIBRATION — are the submitted probabilities honest?

 bin    n  predicted  observed     gap
   0 1000     0.0430     0.079  0.0360
   1 1000     0.0517     0.071  0.0193
   2 1000     0.0568     0.075  0.0182
   3 1000     0.0614     0.090  0.0286
   4 1000     0.0660     0.070  0.0040
   5 1000     0.0709     0.083  0.0121
   6 1000     0.0766     0.075 -0.0016
   7 1000     0.0836     0.081 -0.0026
   8 1000     0.0946     0.083 -0.0116
   9 1000     0.1277     0.079 -0.0487

predicted mean 0.0732 vs base rate 0.0786
Brier 0.0730 vs no-skill 0.0724
-> the probabilities sit at the base rate, which is the correct answer when
   there is nothing to predict.


---
# 8. Business impact estimation

## The cost model

Offering a retention deal costs `C_offer` whether or not the customer would have
churned. Not offering costs `p × C_loss` in expectation. So:

```
offer  iff   p × C_loss > C_offer      ->      p > C_offer / C_loss
```

With the brief's 5:1 ratio the theoretically optimal threshold is `tau* = 1/5 = 0.20`.
Note this is **well above the 7.86% base rate** — which is the whole economic point:
at a 5:1 loss ratio a random customer is *not* worth targeting (expected saving
0.079 × 5 = 0.39 units, against 1 unit of offer cost).

In [11]:
C_OFFER, C_LOSS = 1.0, 5.0
never = s["cost_never_target"]
allc = s["cost_target_all"]
perfect = s["cost_perfect_model"]
model_c = s["cost_model_at_tau_star"]

print(f"Retention offer cost      : {C_OFFER:.0f} unit")
print(f"Cost of losing a customer : {C_LOSS:.0f} units  (brief: 5x retention)")
print(f"Derived optimal threshold : tau* = {s['optimal_tau_theory']:.4f}\n")

print(f"{'strategy':36s} {'cost':>8s} {'vs do-nothing':>14s}")
for label, c in [("Do nothing", never),
                 ("Target everybody", allc),
                 ("Perfect model (upper bound)", perfect),
                 (f"Model @ tau* = {s['optimal_tau_theory']:.2f}", model_c)]:
    print(f"{label:36s} {c:8.0f} {never - c:+14.0f}")

print(f"\nMaximum saving available to ANY model: {s['max_achievable_saving']:.0f} units")
print(f"Value actually captured by the model : {s['value_captured_pct']:+.2%}")
print()
print("-> Targeting everybody is the WORST option: it costs more in offers than the")
print("   churn it prevents. The model does not beat doing nothing, because its")
print("   ranking contains no information.")

Retention offer cost      : 1 unit
Cost of losing a customer : 5 units  (brief: 5x retention)
Derived optimal threshold : tau* = 0.2000

strategy                                 cost  vs do-nothing
Do nothing                               3930             +0
Target everybody                        10000          -6070
Perfect model (upper bound)               786          +3144
Model @ tau* = 0.20                      4039           -109

Maximum saving available to ANY model: 3144 units
Value actually captured by the model : -3.47%

-> Targeting everybody is the WORST option: it costs more in offers than the
   churn it prevents. The model does not beat doing nothing, because its
   ranking contains no information.


In [12]:
print("SENSITIVITY — how the decision changes with the acquisition ratio\n")
sens = pd.read_csv("../outputs/cost_sensitivity.csv")
print(sens.to_string(index=False))
print()
print("-> at every ratio tested the model is never the cheapest policy. A model is")
print("   only worth deploying when it beats both trivial strategies.")

SENSITIVITY — how the decision changes with the acquisition ratio

ratio  theory_tau  never  everybody  model  cheapest
  1:1      1.0000    786      10000 3930.0 everybody
  2:1      0.5000   1572      10000 3934.0 everybody
  3:1      0.3333   2358      10000 3946.0 everybody
  5:1      0.2000   3930      10000 4039.0 everybody
 10:1      0.1000   7860      10000 4596.0 everybody
 20:1      0.0500  15720      10000 6407.0 everybody

-> at every ratio tested the model is never the cheapest policy. A model is
   only worth deploying when it beats both trivial strategies.


### What would the model have to achieve to be worth deploying?

With 786 churners among 10,000 customers:

| Policy | Cost | Comment |
|---|---|---|
| Do nothing | 3,930 units | Lose every churner |
| Target everybody | 10,000 units | Waste 9,214 offers |
| **Perfect model** | **786 units** | Offer only to true churners — the upper bound |
| Our model @ τ* | 4,039 units | **Worse than doing nothing** |

The gap between doing nothing (3,930) and the perfect model (786) is **3,144 units
of addressable value**. The model captures **none of it**. A real retention
programme needs a ranking that is meaningful in the top deciles; this data cannot
produce one.

---
# 9. Bonus challenges

## 9.1 SHAP explainability

If a churn driver existed, SHAP would concentrate attribution on it. Instead the
attribution is spread flat across all features.

In [13]:
shap_imp = pd.read_csv("../outputs/shap_importance.csv")
print("top 12 features by mean |SHAP value|:\n")
print(shap_imp.head(12).round(5).to_string(index=False))
print(f"\ntop feature holds {shap_imp.share.iloc[0]:.2%} of total attribution")
print("-> no dominant driver. A signal-free label produces exactly this pattern:")
print("   many features carrying small, mutually-cancelling contributions.")

top 12 features by mean |SHAP value|:

                 feature  mean_abs_shap   share
          call_halfratio        0.00304 0.03420
                  sms_sd        0.00299 0.03370
              data_slope        0.00265 0.02979
complaints_per_call_hour        0.00264 0.02977
                 data_cv        0.00261 0.02942
           call_trend_r2        0.00259 0.02912
    MonthlyCharges_clean        0.00239 0.02689
                data_max        0.00235 0.02647
   call_last_minus_first        0.00221 0.02489
          data_rel_slope        0.00217 0.02446
   data_last_minus_first        0.00205 0.02312
              sms_median        0.00205 0.02310

top feature holds 3.42% of total attribution
-> no dominant driver. A signal-free label produces exactly this pattern:
   many features carrying small, mutually-cancelling contributions.


## 9.2 Data drift detection

Population Stability Index compares each month against a months 1–6 reference
(< 0.1 stable, 0.1–0.25 moderate, > 0.25 major shift).

In [14]:
drift = pd.read_csv("../outputs/drift_psi.csv")
print(drift.round(4).to_string(index=False))
base = drift.loc[drift.month <= 9, ["call", "data"]].max().max()
step = drift.loc[drift.month >= 10, ["call", "data"]].min().min()
print(f"\nmonths 1-9  max PSI (call/data): {base:.4f}")
print(f"months 10-12 min PSI (call/data): {step:.4f}   -> a {step/base:.0f}x step change")
print()
print("-> CallMinutes and DataUsageGB both step up at month 10 while SMSCount and")
print("   Complaints stay flat. The absolute PSI stays under the conventional 0.1")
print("   alarm, so this is a small but unambiguous structural break rather than a")
print("   major drift event - still worth flagging, because a model trained on")
print("   months 1-9 faces a measurably different input distribution from month 10.")

 month   call   data    sms   comp
     1 0.0013 0.0012 0.0010 0.0001
     2 0.0004 0.0004 0.0008 0.0001
     3 0.0007 0.0004 0.0016 0.0000
     4 0.0008 0.0007 0.0003 0.0001
     5 0.0005 0.0015 0.0006 0.0001
     6 0.0007 0.0004 0.0012 0.0000
     7 0.0008 0.0010 0.0009 0.0007
     8 0.0014 0.0006 0.0013 0.0000
     9 0.0008 0.0007 0.0007 0.0001
    10 0.0465 0.0270 0.0002 0.0000
    11 0.0462 0.0342 0.0007 0.0002
    12 0.0487 0.0301 0.0003 0.0001

months 1-9  max PSI (call/data): 0.0015
months 10-12 min PSI (call/data): 0.0270   -> a 18x step change

-> CallMinutes and DataUsageGB both step up at month 10 while SMSCount and
   Complaints stay flat. The absolute PSI stays under the conventional 0.1
   alarm, so this is a small but unambiguous structural break rather than a
   major drift event - still worth flagging, because a model trained on
   months 1-9 faces a measurably different input distribution from month 10.


## 9.3 Cost-sensitive learning

Two ways to make a learner cost-aware: reweight the training samples, or move the
decision threshold. Both are tested against the cost model.

In [15]:
csl = pd.read_csv("../outputs/cost_sensitive_learning.csv")
print(csl.round(4).to_string(index=False))
print()
print("-> reweighting shifts the threshold the model prefers but not its ranking power")
print("   (ROC-AUC is invariant to the decision threshold). It cannot recover value")
print("   that is absent from the features.")

               variant  roc_auc  best_tau  min_cost  vs_never  pct_of_achievable
        plain logistic   0.4964    0.4237    3930.0       0.0                0.0
cost-weighted logistic   0.4958    0.7733    3930.0       0.0                0.0

-> reweighting shifts the threshold the model prefers but not its ranking power
   (ROC-AUC is invariant to the decision threshold). It cannot recover value
   that is absent from the features.


## 9.4 Survival analysis — feasibility assessment

**Conclusion: not feasible on this dataset**, and the reason is structural rather
than a limitation of effort.

In [16]:
print("Available columns")
print(f"  churn_labels : {list(cl.columns)}")
print(f"  usage_data   : {list(ud.columns)}")
print(f"  target values: {sorted(cl.Churn.unique())}")
print()
print("A survival model needs, per subject:")
print("  (a) a duration        - months from origin to the churn event")
print("  (b) an event indicator - observed churn vs right-censored")
print()
print("This dataset provides NEITHER. There is no churn date and no tenure field.")
print("Every churner would receive the same duration (the end of the 12-month window)")
print("and every non-churner is censored at that same instant, so the hazard function")
print("is not identifiable.")
print()
print("What would be needed: a churn date or churn month per customer, plus tenure at")
print("observation start.")

Available columns
  churn_labels : ['CustomerID', 'Churn']
  usage_data   : ['CustomerID', 'Month', 'CallMinutes', 'DataUsageGB', 'SMSCount', 'Complaints']
  target values: [np.int64(0), np.int64(1)]

A survival model needs, per subject:
  (a) a duration        - months from origin to the churn event
  (b) an event indicator - observed churn vs right-censored

This dataset provides NEITHER. There is no churn date and no tenure field.
Every churner would receive the same duration (the end of the 12-month window)
and every non-churner is censored at that same instant, so the hazard function
is not identifiable.

What would be needed: a churn date or churn month per customer, plus tenure at
observation start.


---
# 10. Conclusion

## What was delivered

| Requirement | Status |
|---|---|
| Clean and preprocess the messy datasets | 11 defects, one documented rule each, every change flagged |
| Engineer meaningful behavioural features | 66 features in 6 families, including the full trend family |
| Build a model estimating churn probability | 5 candidates, out-of-fold probabilities, `prediction.csv` submitted |
| Align evaluation with business impact | Cost model with derived threshold, sensitivity analysis, value-captured metric |
| Bonus: SHAP | Completed — attribution flat, no dominant driver |
| Bonus: drift detection | Completed — `CallMinutes` breaks at month 10 |
| Bonus: cost-sensitive learning | Completed — reweighting cannot fix an uninformative ranking |
| Bonus: survival analysis | Assessed — not feasible, no event time or censoring indicator |

## The finding that matters

**The churn label carries no learnable signal from the provided features.**

- Best model ROC-AUC **0.5181** against a shuffled-label null band of 0.4772–0.5198
  (10% of null draws matched or beat it)
- PR-AUC **0.0808** against a base rate of 0.0786 — no lift
- Top-decile lift **0.95x** — the "highest risk" decile churns *less* than average
- SHAP attribution flat across all 70 features
- The cost-optimal policy is to send **no** retention offers at all

This is not a modelling failure. It is a property of the data, confirmed by four
independent methods, plus direct verification in section 4 that the brief's claimed
"usage decline before churn" mechanism is absent.

## Recommendations

1. **Do not deploy churn targeting on this data.** Any model would be
   indistinguishable from randomly selecting customers.
2. **Fix the source data first** — DQ-07 makes every tenure-based feature
   unreliable for 751 customers.
3. **Resolve the DD/MM vs MM/DD ambiguity at the ingestion layer.** It is systemic,
   not confined to the 5 rows where a duplicate happened to expose it.
4. **Investigate the month-10 structural break** in `CallMinutes` before building
   any time-series feature on it.
5. **Treat the label as suspect.** A signal-free target plus Gaussian/uniform
   synthetic usage metrics suggests churn was assigned independently of the
   features during dataset construction.

## The professional point

The deliverable of this notebook is a *refusal* — evidence, assembled in hours
rather than weeks, that a churn model should not be shipped. Reporting that a model
cannot work is more valuable than reporting one that appears to work and quietly
fails in production.